# Offline evaluation on `shop.history_sessions`

Before putting a new recommender live, estimate offline whether it beats the baseline.

The log was produced by the **old system** (best sellers plus a bit of randomness), so it only tells you what happened with the products *it* showed. Two estimates:

1. **Hit rate**: of the impressions where the shopper clicked, how often is the clicked product in your top 10? Easy, but biased towards popular products (the old system showed them most).
2. **Inverse propensity scoring (IPS)** of the click rate in **slot 1**: keep only the impressions where your first product equals the one the old system put in slot 1, and weight each by `1 / propensity`. This corrects the popularity bias of the log for that slot. SNIPS (self-normalised IPS) has lower variance.

We hold out the **last 7 days** of the history: the policies are built only from data before the cut and evaluated after it.

In [31]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests


pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q):
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

## Data

In [32]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased, revenue_eur
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

train <  2026-11-07 <= test
100000 test impressions, observed CTR 0.0376


,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased,revenue_eur
0,r_6f1580f4de85c0a2,2026-11-11 03:21:52.345000000,u_dbac4031,"{'device': 'ios', 'page': 'product', 'region': 'Aragón'}","[{'product_id': 'p_e5796b', 'ts': '2026-11-11 03:18:40.326000000'}]","[p_6d0f23, p_700c33, p_dbd2ba, p_73f595, p_57550f, p_bdb564, p_4e510b, p_0390e6, p_6f69a8, p_1bcbe1]","[0.021461, 0.002256, 0.01923, 0.065452, 0.033274, 0.018341, 0.01166, 0.014878, 0.008002, 0.205608]",NaN,NaN,0.0
1,r_01b41a6acbaae0a7,2026-11-11 04:51:29.967000000,u_522c2992,"{'device': 'android', 'page': 'home', 'region': 'Catalunya'}",[],"[p_dc1c2e, p_3c342d, p_8923f4, p_e2454f, p_a187f7, p_7eb151, p_9265c4, p_a9b2bd, p_0ab8c3, p_1a7fd6]","[0.078911, 0.088282, 0.00955, 0.360097, 0.019411, 0.035589, 0.002165, 0.07204, 0.009168, 0.051751]",NaN,NaN,0.0
2,r_2095727203e8f5b4,2026-11-13 15:51:57.897000000,u_88d36cec,"{'device': 'web', 'page': 'home', 'region': 'Canarias'}",[],"[p_92d45d, p_bc80d2, p_2c88d5, p_64d53d, p_4ba423, p_14c2d9, p_8ca8ed, p_e2c959, p_d72d81, p_d75243]","[0.029457, 0.047013, 0.239676, 0.02507, 0.022157, 0.000331, 0.326602, 0.198753, 0.039119, 0.023574]",NaN,NaN,0.0


In [33]:
import sys
sys.path.insert(0, ".")
from recommender import best_sellers, recommend   # the same code that runs live

best = best_sellers(until=CUT)

## Policies

A policy maps one request (a row) to an ordered list of 10 product IDs.

In [34]:
def baseline(row):
    """The live baseline, imported from starter/recommender.py."""
    return recommend(row.recent_views, best)


POLICIES = {"baseline": baseline}

## Evaluation

In [35]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    prop = pd.to_numeric(
    df.propensity.str[0],
        errors="coerce"
    ).to_numpy(dtype=float)
    
    valid = np.isfinite(prop) & (prop > 0)
    
    # Si una recomendación coincide con una propensión inválida,
    # no podemos estimar su recompensa mediante IPS.
    if np.any(match & ~valid):
        raise ValueError(
            f"{np.sum(match & ~valid)} coincidencias "
            "con propensión inválida"
        )
    
    w = np.zeros(len(df), dtype=float)
    
    np.divide(
        match.astype(float),
        prop,
        out=w,
        where=match & valid
    )
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

old system, observed slot-1 CTR: 0.0111


KeyboardInterrupt: 

## Next steps

- Few `slot1_matches` means a noisy IPS estimate: report it next to the estimate.
- Personal taste: who bought what in `shop.history_events` (matrix factorisation, item-to-item similarity).
- Cold start: new products only have a `description`. Text embeddings can place them next to similar products.
- Context: does price sensitivity change with `device` and `region`?
- Live: the stream changes (trends, price drops, stock-outs). Your `team_NN.feedback` tells you how you are really doing.

# TEST

In [36]:
sql("""
SELECT
    s.model_version::String AS model,

    min(f.ts::DateTime64(3)) AS first_request,
    max(f.ts::DateTime64(3)) AS last_request,

    count() AS requests,

    countIf(f.clicked::String != '') AS clicks,
    countIf(f.purchased::String != '') AS purchases,

    round(
        100.0 * countIf(f.clicked::String != '') / count(), 3
    ) AS ctr_pct,

    round(
        100.0 * countIf(f.purchased::String != '') / count(), 3
    ) AS conversion_pct,

    round(
        sum(f.revenue_eur::Float64) / count(), 4
    ) AS eur_per_request,

    round(avg(f.latency_ms::Float64), 1) AS latency_ms

FROM team_04.feedback AS f

INNER JOIN team_04.submissions AS s
    ON f.request_id::String = s.request_id::String

WHERE f.request_id != '__init__'
    AND f.served_by::String = 'team'
    AND s.model_version::String IN (
        'item2vec-hybrid-0.2',
        'lightgbm-lambdarank-v1'
    )

GROUP BY model
ORDER BY conversion_pct DESC
""")

,model,first_request,last_request,requests,clicks,purchases,ctr_pct,conversion_pct,eur_per_request,latency_ms
0,item2vec-hybrid-0.2,2026-11-14 16:47:57.951,2026-11-14 18:01:42.575,13124,244,65,1.859,0.495,0.1133,1388.8
1,lightgbm-lambdarank-v1,2026-11-14 17:55:23.550,2026-11-14 18:01:42.575,1137,27,3,2.375,0.264,0.0281,1241.1


In [ ]:
sql("""
    SELECT
        toStartOfMinute(ts::DateTime64(3)) AS minute,
        count() AS total,
        countIf(served_by::String = 'team') AS team,
        countIf(
            served_by::String = 'fallback'
            AND fallback_reason::String = 'missing'
        ) AS missing,
        round(
            100.0 * countIf(served_by::String = 'team') / count(),
            2
        ) AS success_pct
    FROM team_04.feedback
    WHERE request_id != '__init__'
      AND ts::DateTime64(3) >= now() - INTERVAL 10 MINUTE
    GROUP BY minute
    ORDER BY minute DESC
""")

## Download data

In [ ]:
df = sql(f"""
    SELECT *
    FROM shop.history_sessions  
    ORDER BY rand() 
    LIMIT 100000
""")

df.to_csv("../data/history_sessions.csv", index=False)

In [ ]:
df = sql(f"""
    SELECT *
    FROM shop.catalog  
    ORDER BY rand() 
    LIMIT 100000
""")

df.to_csv("../data/catalog.csv", index=False)

In [ ]:
df = sql(f"""
    SELECT *
    FROM shop.history_events  
    ORDER BY rand() 
    LIMIT 100000
""")

df.to_csv("../data/history_events.csv", index=False)

In [ ]:
sql(f"""
    SELECT *
    FROM shop.catalog_updates  
    ORDER BY rand() 
    LIMIT 100000
""")

df.to_csv("../data/catalog_updates.csv", index=False)

## Duckdb

In [ ]:
import duckdb

con = duckdb.connect()

result = con.sql("""
SELECT
    list_sum(propensity::DOUBLE[]) AS total
FROM read_csv_auto('../data/history_sessions.csv')
WHERE clicked != 'NC'
LIMIT 10;
""")

print(result.show())

## Pandas

In [ ]:
path = "../data/history_sessions.csv"

# Leer CSV
df = pd.read_csv(path)

# Sustituir valores nulos, "NA" y strings vacíos
df["clicked"] = df["clicked"].replace(["", "NA"], pd.NA).fillna("NC")
df["purchased"] = df["purchased"].replace(["", "NA"], pd.NA).fillna("NP")
df["ts_epoch"] = pd.to_datetime(df["ts"], utc=True).astype("int64") 

# Sobrescribir el CSV original
df.to_csv(path, index=False)

In [ ]:
catalogs = pd.read_csv("../data/catalog.csv")

## SQL

In [ ]:
sql(f"""
    SELECT user_id, context, clicked, purchased, revenue_eur, 
    event_type, price_eur, device, region, 
    category, subcategory
    FROM shop.history_sessions
""")

In [ ]:

df = sql(f"""
    SELECT *
    FROM shop.history_sessions
    where user_id = 'u_99887b61' 
    -- and ts = '2026-09-19 19:28:34.567000000'
    order by ts
""")

df

In [ ]:

dfe = sql(f"""
    SELECT *
    FROM shop.history_events
    where user_id = 'u_99887b61' 
    -- and ts = '2026-09-19 19:28:34.567000000'
    --and product_id = 'p_de0a82'
    order by ts
""")

dfe


In [ ]:
user_history = sql(f"""
    SELECT
        user_id::String AS user_id,
        arraySlice(
            arrayMap(
                x -> x.2,
                arraySort(
                    x -> x.1,
                    groupArray((
                        ts::DateTime64(3),
                        product_id::String
                    ))
                )
            ),
            -20
        ) AS products
    FROM shop.history_events
    WHERE ts < '{CUT}'
      AND event_type IN ('purchase', 'add_to_cart')
      AND user_id IS NOT NULL
      AND product_id IS NOT NULL
    GROUP BY user_id
""")

## W2V

In [ ]:
sessions = []

for bucket in range(32):
    df = sql(f"""
        SELECT
            session_id::String AS session_id,
            arrayMap(
                x -> x.2,
                arraySort(
                    x -> x.1,
                    groupArray((
                        ts::DateTime64(3),
                        product_id::String
                    ))
                )
            ) AS products
        FROM shop.history_events
        WHERE ts < '{CUT}'
          AND session_id IS NOT NULL
          AND product_id IS NOT NULL
          AND cityHash64(session_id::String) % 32 = {bucket}
        GROUP BY session_id
        HAVING uniqExact(product_id::String) >= 2
    """)

    sessions.extend(df["products"].tolist())
    print(f"Partición {bucket + 1}/32: {len(sessions)} sesiones")

In [ ]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=sessions,
    vector_size=64,
    window=5,
    min_count=2,
    sg=1,
    workers=4,
    epochs=10
)

print("Sesiones:", len(sessions))
print("Productos con embedding:", len(model.wv))

In [ ]:
model.save("../data/item2vec.model")

In [ ]:
product_id = sessions[0][0]

if product_id in model.wv:
    print(model.wv.most_similar(product_id, topn=10))

In [ ]:
from collections import defaultdict
from functools import lru_cache

pop_rank = {pid: i for i, pid in enumerate(best)}

@lru_cache(maxsize=10000)
def item_neighbors(pid):
    if pid not in model.wv:
        return ()
    return tuple(model.wv.most_similar(pid, topn=80))


def hybrid_v2(recent_views, alpha=0.2, mode="sum"):
    views = [
        v["product_id"]
        for v in (recent_views or [])
        if isinstance(v, dict) and v.get("product_id")
    ]

    if not views:
        return best[:10]

    # Últimas 3 visitas
    views = views[-3:]

    if mode == "last":
        views = views[-1:]

    scores = defaultdict(float)

    for age, pid in enumerate(reversed(views)):
        weight = 0.7 ** age

        candidates = [(pid, 1.0)] if pid in model.wv else []
        candidates += item_neighbors(pid)

        for neighbor, similarity in candidates:
            value = weight * max(0, similarity)

            if mode == "max":
                scores[neighbor] = max(scores[neighbor], value)
            else:
                scores[neighbor] += value

    emb_rank = {
        pid: i
        for i, (pid, score) in enumerate(
            sorted(scores.items(), key=lambda x: (-x[1], x[0]))
        )
    }

    candidates = set(best[:200]) | set(scores)

    def score(pid):
        pop = (
            1 / (30 + pop_rank[pid])
            if pid in pop_rank else 0
        )

        emb = (
            1 / (30 + emb_rank[pid])
            if pid in emb_rank else 0
        )

        return (1 - alpha) * pop + alpha * emb

    ranked = sorted(
        candidates,
        key=lambda pid: (-score(pid), pid)
    )

    # Completar con popularidad y eliminar duplicados
    result = list(dict.fromkeys(ranked + best))
    return result[:10]

In [ ]:
validation = test.sample(
    n=100000,
    random_state=42
)

results = {}

for alpha in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:

    def policy(row):
        return hybrid_recommend(
            row.recent_views,
            alpha=alpha
        )

    results[f"hybrid_{alpha}"] = evaluate(
        policy,
        validation
    )

pd.DataFrame(results).T

In [ ]:
import numpy as np

def ips_diagnostics(policy, df):
    recs = [policy(r) for r in df.itertuples()]

    first_shown = df.shown.str[0].to_numpy()

    match = np.array([
        r[0] == s
        for r, s in zip(recs, first_shown)
    ])

    prop = df.propensity.str[0].to_numpy().astype(float)

    valid = np.isfinite(prop) & (prop > 0)

    w = np.zeros(len(df))
    w[match & valid] = 1.0 / prop[match & valid]

    ess = (
        w.sum() ** 2 / (w ** 2).sum()
        if (w ** 2).sum() > 0 else 0
    )

    return {
        "matches": int(match.sum()),
        "ESS": round(ess, 2),
        "max_weight": round(w.max(), 2),
        "mean_weight": round(w.mean(), 4)
    }

In [ ]:
results = {}

for alpha in [0.0, 0.2, 0.4]:

    def policy(row, a=alpha):
        return hybrid_recommend(
            row.recent_views,
            alpha=a
        )

    metrics = evaluate(policy, test)
    diagnostics = ips_diagnostics(policy, test)

    results[alpha] = {
        **metrics,
        **diagnostics
    }

pd.DataFrame(results).T

In [ ]:
import numpy as np
import pandas as pd

def evaluate_purchases(policy, df):
    recs = [policy(r) for r in df.itertuples()]

    first_shown = df.shown.str[0].to_numpy()

    match = np.array([
        len(r) > 0 and r[0] == s
        for r, s in zip(recs, first_shown)
    ])

    purchased = df.purchased.to_numpy()

    reward = (
        (purchased == first_shown)
        & pd.notna(purchased)
    ).astype(float)

    prop = df.propensity.str[0].to_numpy(dtype=float)

    valid = match & np.isfinite(prop) & (prop > 0)

    w = np.zeros(len(df))
    w[valid] = 1 / prop[valid]

    return {
        "IPS_purchase": (w * reward).mean(),
        "SNIPS_purchase": (
            (w * reward).sum() / max(w.sum(), 1e-9)
        ),
        "matched_purchases": int(
            (match & (reward == 1)).sum()
        ),
        "ESS": (
            w.sum() ** 2 / (w ** 2).sum()
            if (w ** 2).sum() > 0 else 0
        )
    }

In [ ]:
results_purchase = {}

for alpha in [0.0, 0.2, 0.4]:

    def policy(row, a=alpha):
        return hybrid_recommend(
            row.recent_views,
            alpha=a
        )

    results_purchase[alpha] = evaluate_purchases(
        policy,
        test
    )

pd.DataFrame(results_purchase).T

In [ ]:
print("Sesiones de entrenamiento:", len(sessions))
print("Total interacciones:", sum(len(s) for s in sessions))
print("Productos con embeddings:", len(model.wv))
print("Media de productos por sesión:",
      sum(len(s) for s in sessions) / len(sessions))

In [ ]:
POLICIES = {
    "hybrid_original": lambda r: hybrid_recommend(
        r.recent_views, alpha=0.2
    ),
    "item2vec_sum": lambda r: hybrid_v2(
        r.recent_views, alpha=0.2, mode="sum"
    ),
    "item2vec_max": lambda r: hybrid_v2(
        r.recent_views, alpha=0.2, mode="max"
    ),
    "item2vec_last": lambda r: hybrid_v2(
        r.recent_views, alpha=0.2, mode="last"
    )
}

pd.DataFrame({
    name: evaluate(policy, test)
    for name, policy in POLICIES.items()
}).T

In [ ]:
prop = pd.to_numeric(
    test.propensity.str[0],
    errors="coerce"
)

print("NULL:", prop.isna().sum())
print("Ceros:", (prop == 0).sum())
print("Negativos:", (prop < 0).sum())
print("Mínimo:", prop.min())
print("Máximo:", prop.max())

## LightGBM

In [ ]:
import pandas as pd
from lightgbm_ranker import ShopRanker, sample_history_csv

# Cargar una muestra del histórico, sin cargar todo en memoria
history = sample_history_csv(
    "../data/history_sessions.csv",
    fraction=1.0
)

# Excluir el período de evaluación
history = history[
    history["ts"] < pd.Timestamp(CUT, tz="UTC")
]

# Crear el ranker
ranker = ShopRanker(
    catalog=catalogs,
    best=best,
    item2vec=model
)

# Entrenar
ranker.fit(history)

In [ ]:
policies = {
    "baseline": lambda r: best[:10],
    "hybrid_0.2": lambda r: hybrid_v2(
        r.recent_views,
        alpha=0.2
    ),
    "lightgbm": ranker.policy
}

results = {}

for name, policy in policies.items():
    metrics = evaluate(policy, test)
    purchases = evaluate_purchases(policy, test)

    results[name] = {
        **metrics,
        **purchases
    }

pd.DataFrame(results).T

In [ ]:
import time
import numpy as np

sample = test.sample(1000, random_state=42)

latencies = []

for row in sample.itertuples():
    start = time.perf_counter()
    ranker.policy(row)
    latencies.append((time.perf_counter() - start) * 1000)

print(f"Media: {np.mean(latencies):.2f} ms")
print(f"P95: {np.percentile(latencies, 95):.2f} ms")
print(f"P99: {np.percentile(latencies, 99):.2f} ms")

In [ ]:
ranker.save("../data/lgbm_ranker.joblib")